# CTE FAGEN Reproduction

Trains the contrastive trajectory encoder on the paper's training split (10 contexts in $g\in[5.0, 9.74]$ m/s$^2$), embeds held-out contexts $g\in\{3, 4, 12, 15\}$, and computes the **manifold-projection error** number that the FAGEN reviewer asked for.

**Outputs you'll get from this notebook:**
1. `projection_error.json` — within-training mean residual + per-held-out-$g$ L2 residual (the number to drop into §5.2 of the paper)
2. `fig5_latent_space_real.png` — t-SNE of real embeddings (replaces the simulated figure in the paper)
3. `embeddings.npz` — raw 64-D embeddings for downstream analysis
4. `encoder.pt` — trained encoder checkpoint

**Recommended runtime:** Colab T4 GPU, ~15-30 min total.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/varchanaiyer/cte-fagen-reproduction.git
%cd cte-fagen-reproduction

## 2. Install dependencies

On Colab this takes ~2 minutes. The `gym` deprecation warning from CARL is expected and harmless.

In [ ]:
!pip install -q -r requirements.txt

## 3. Detect device

Should print `cuda` on Colab GPU runtimes, `mps` on Apple Silicon laptops, `cpu` otherwise. CPU works but is slow (~30 min vs ~5 min on GPU).

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'PyTorch device: {device}')
print(f'Torch version : {torch.__version__}')
if device == 'cuda':
    print(f'GPU           : {torch.cuda.get_device_name(0)}')

## 4. (Optional) Sanity-check the encoder import

Catches setup issues before kicking off the long-running training cell.

In [ ]:
from src.models import LSTMEncoder, SupConLoss
from src.data import TrajectoryCollector, get_context_distributions
from src.training import EncoderTrainer
print('imports OK')

# Quick smoke test: 1 segment per context, 1 epoch, just to confirm the
# data collection + forward pass run without errors.
import numpy as np
tiny_contexts = [{'g': 5.0}, {'g': 7.0}, {'g': 9.0}]
smoke = TrajectoryCollector(env_name='pendulum', contexts=tiny_contexts,
                            segment_length=32, num_segments_per_context=2,
                            policy='random', seed=0)
smoke_segs = smoke.collect(verbose=False)
print(f'smoke test collected {len(smoke_segs)} segments')
obs_dim = smoke_segs[0].observations.shape[-1]
act_dim = smoke_segs[0].actions.shape[-1] if smoke_segs[0].actions.ndim > 1 else 1
enc = LSTMEncoder(input_dim=obs_dim+act_dim, hidden_dim=64, latent_dim=64).to(device)
x = torch.randn(2, 32, obs_dim+act_dim).to(device)
z = enc(x)
print(f'encoder forward OK, output shape = {tuple(z.shape)}')

## 5. Run the diagnostics pipeline

This is the long-running cell. It:
1. Collects 100 segments per training context (10 contexts) and 30 segments per held-out context (4 contexts).
2. Trains the BiLSTM encoder for 50 epochs with InfoNCE/SupCon loss.
3. Embeds train + held-out segments (64-D each).
4. Fits a 1-D principal axis through training-context centroids and computes L2 projection residuals.
5. Writes results to `diagnostics_outputs/`.

Expected wall-clock on Colab T4: ~10-15 min.

**If you want sharper embeddings**, edit `compute_diagnostics.py` and bump `NUM_EPOCHS` to 100 or 200 before running.

In [ ]:
!python compute_diagnostics.py

## 6. Read the projection-error number

This is **the value to drop into §5.2 of the paper** (replacing the artifact-bundle deferral).

In [ ]:
import json
with open('diagnostics_outputs/projection_error.json') as f:
    diag = json.load(f)

print('=== Manifold-projection error (real, single-seed) ===')
print(f"within-training mean residual : {diag['within_train_mean_residual']:.4f}")
for g, info in diag['heldout'].items():
    print(f"  g = {float(g):>5.1f}  L2 residual = {info['mean_l2_residual']:.4f}  "
          f"(n={info['n_segments']}, std={info['std']:.4f})")
print()
print('Suggested §5.2 sentence (paste verbatim into main.tex):')
g_low_keys  = sorted([k for k in diag['heldout'] if float(k) < 5.0])
g_high_keys = sorted([k for k in diag['heldout'] if float(k) > 10.0])
g_low_str  = ', '.join(f"g={float(k):.0f}: {diag['heldout'][k]['mean_l2_residual']:.3f}" for k in g_low_keys)
g_high_str = ', '.join(f"g={float(k):.0f}: {diag['heldout'][k]['mean_l2_residual']:.3f}" for k in g_high_keys)
print(f'  Held-out projection residuals (mean L2 vs training-context line) — '
      f"low-g [{g_low_str}], high-g [{g_high_str}]; "
      f"compare with within-training mean of {diag['within_train_mean_residual']:.3f}.")

## 7. Inspect the real t-SNE figure

This is the figure to replace `fig5_latent_space.png` in the paper if you want measured rather than simulated embeddings.

In [ ]:
from IPython.display import Image
Image('diagnostics_outputs/fig5_latent_space_real.png')

## 8. Download the outputs

On Colab, this triggers a download dialog for each file. Locally this is a no-op.

In [ ]:
try:
    from google.colab import files
    files.download('diagnostics_outputs/projection_error.json')
    files.download('diagnostics_outputs/fig5_latent_space_real.png')
    files.download('diagnostics_outputs/embeddings.npz')
    files.download('diagnostics_outputs/encoder.pt')
except ImportError:
    print('not in Colab; outputs are in ./diagnostics_outputs/')

## 9. (Optional) What to do with the outputs

**Paste into the paper's §5.2** (replacing the existing "projection-error values for $g\in\{12,15\}$ are reported in the artifact bundle" sentence):

```latex
Held-out embeddings have mean L2 residual to the predicted line position
of <X> at $g{=}12$ and <Y> at $g{=}15$ in 64-D latent space, compared to
a within-training mean residual of <Z>. The held-out values are within
<W>x of within-training noise, supporting the claim that the encoder
places high-gravity contexts on the predicted manifold direction.
```

Replace `<X>`, `<Y>`, `<Z>`, `<W>` with the numbers from cell 6.

**Replace the simulated `fig5_latent_space.png`** with `diagnostics_outputs/fig5_latent_space_real.png` and update the figure caption to: `Real CTE latent embeddings (single seed, 50-epoch encoder, n=100 segments per training context, n=30 per held-out context).`